# Radial 2D GRE: simulation and self-supervised DeepADMM reconstruction

Pipeline: (1) build a golden-angle-free radial 2D GRE sequence with Pulseq,
(2) simulate the multi-coil signal on a numerical phantom, (3) reconstruct with
conjugate gradient (CG) and with a self-supervised, learned ADMM (DeepADMM).

## Imports and device

All tensors are created on the GPU by default.

In [ ]:
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
import torch.nn as nn
from tqdm import tqdm

from Modules.CoilSens import cyclic_head_coil
from Modules.fast_iso import isochromat_sim_fast
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import IdentityOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.Regularizer import Regularizer

torch.set_default_device("cuda")

## Sequence design

Radial spoiled GRE with `n_dummy` dummy shots at angle 0 (no ADC) to reach steady
state, followed by `n_spokes // undersampling_factor` spokes spread over 180 degrees.

In [ ]:
def main(
    plot: bool, write_seq: bool, seq_filename: str = "gre_radial_pypulseq.seq"
) -> None:
    """Build, check and optionally plot/write the radial 2D GRE sequence.

    Parameters
    ----------
    plot : bool
        Plot the sequence after the timing check.
    write_seq : bool
        Write the sequence to ``Radial 2D GRE.seq`` (read by the following cells).
    seq_filename : str, optional
        Currently unused; the output file name is fixed to ``Radial 2D GRE.seq``.

    Returns
    -------
    None
    """
    seq = pp.Sequence()
    fov = 260e-3
    n_x = 128
    flip_angle_deg = 5
    undersampling_factor = 4
    slice_thickness = 3e-3
    echo_time = 4e-3
    repetition_time = 100e-3
    n_spokes = 256
    n_dummy = 20

    system = pp.Opts(
        max_grad=28,
        grad_unit="mT/m",
        max_slew=120,
        slew_unit="T/m/s",
        rf_ringdown_time=20e-6,
        rf_dead_time=100e-6,
        adc_dead_time=10e-6,
    )

    rf, grad_slice, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=flip_angle_deg * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
    )

    delta_k = 1 / fov
    grad_readout = pp.make_trapezoid(
        channel="x", flat_area=n_x * delta_k, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=n_x,
        duration=grad_readout.flat_time,
        delay=grad_readout.rise_time,
        system=system,
    )
    grad_readout_prephase = pp.make_trapezoid(
        channel="x", area=-grad_readout.area / 2, duration=2e-3, system=system
    )
    grad_slice_rephase = pp.make_trapezoid(
        channel="z", area=-grad_slice.area / 2, duration=2e-3, system=system
    )
    grad_readout_spoil = pp.make_trapezoid(
        channel="x", area=0.5 * n_x * delta_k, system=system
    )
    grad_slice_spoil = pp.make_trapezoid(
        channel="z", area=4 / slice_thickness, system=system
    )

    raster = seq.grad_raster_time
    delay_te = (
        np.ceil(
            (
                echo_time
                - pp.calc_duration(grad_readout_prephase)
                - grad_slice.fall_time
                - grad_slice.flat_time / 2
                - pp.calc_duration(grad_readout) / 2
            )
            / raster
        )
        * raster
    )
    delay_tr = (
        np.ceil(
            (
                repetition_time
                - pp.calc_duration(grad_readout_prephase)
                - pp.calc_duration(grad_slice)
                - pp.calc_duration(grad_readout)
                - delay_te
            )
            / raster
        )
        * raster
    )
    assert delay_tr > pp.calc_duration(grad_readout_spoil, grad_slice_spoil)

    angles = np.linspace(0, np.pi, n_spokes // undersampling_factor, endpoint=False)

    for i in range(-n_dummy, n_spokes // undersampling_factor):
        angle = 0 if i < 0 else angles[i]

        seq.add_block(rf, grad_slice)
        seq.add_block(
            *pp.rotate(grad_readout_prephase, grad_slice_rephase, angle=angle, axis="z")
        )
        seq.add_block(pp.make_delay(delay_te))
        if i > 0:
            seq.add_block(*pp.rotate(grad_readout, adc, angle=angle, axis="z"))
        else:
            seq.add_block(*pp.rotate(grad_readout, angle=angle, axis="z"))
        seq.add_block(
            *pp.rotate(
                grad_readout_spoil,
                grad_slice_spoil,
                pp.make_delay(delay_tr),
                angle=angle,
                axis="z",
            )
        )

    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed! Error listing follows:")
        print(error_report)

    if plot:
        seq.plot()

    if write_seq:
        seq.set_definition("FOV", [fov, fov, slice_thickness])
        seq.set_definition("Name", "gre_rad")
        seq.write("Radial 2D GRE")


if __name__ == "__main__":
    main(plot=True, write_seq=True)

## Phantom and coil sensitivities

The simulation uses coil sensitivities scaled to the physical FOV; the
reconstruction uses unscaled (and flipped) sensitivities on the image grid.

In [ ]:
n_x = 128
n_channels = 8
fov = 260e-3

phantom = mr0.util.load_phantom(size=(n_x, n_x))

coil_sensitivities_recon = cyclic_head_coil((n_channels, n_x, n_x, 1)).to(
    torch.complex64
)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_x, 1),
    fov_scaling=(phantom.size[0] / fov, phantom.size[1] / fov, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

voxel_data = phantom.build()

## Signal simulation

Ground-truth time-domain spin simulation of the sequence written above.

In [ ]:
mr0_seq = mr0.Sequence.import_file("Radial 2D GRE.seq")
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr0_seq,
    voxel_data.cuda(),
    spin_count=256,
    spin_dist="rand",
    print_progress=True,
)

## DeepADMM

Solves $\min_x \tfrac{1}{2}\|Ax - d\|_2^2 + R(x)$ by ADMM, where the $x$-update is
a few CG iterations on $(Q + \rho I)\,x = b + \rho\,(z - u)$ with
$Q = A^H A$, $b = A^H d$, and the $z$-update applies the learned regularizer.
$\rho$ is a trainable parameter.

In [ ]:
class DeepADMM(nn.Module):
    """ADMM solver with a learned regularizer and a trainable penalty weight.

    Parameters
    ----------
    shape : Tuple[int, ...]
        Shape of the image the regularizer and identity operator act on.
    rho_init : float, optional
        Initial value of the ADMM penalty parameter ``rho``.
    maxiter_admm : int, optional
        Number of ADMM iterations.
    maxiter_cg : int, optional
        Number of CG iterations per ADMM ``x``-update.
    """

    def __init__(
        self,
        shape: Tuple[int, ...],
        rho_init: float = 0.1,
        maxiter_admm: int = 10,
        maxiter_cg: int = 5,
    ) -> None:
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(shape)
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg

    def forward(
        self,
        Q: LinearOperator,
        b: torch.Tensor,
        x: torch.Tensor,
        disable_progressbar: bool = True,
    ) -> torch.Tensor:
        """Run the unrolled ADMM iterations.

        Parameters
        ----------
        Q : LinearOperator
            Normal operator ``A^H A``.
        b : torch.Tensor
            Right-hand side ``A^H d`` (real-valued view of the complex image).
        x : torch.Tensor
            Initial image estimate (real-valued view).
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        torch.Tensor
            Reconstructed image (real-valued view).
        """
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)

        for _ in range(self.maxiter_admm):
            x = conjugate_gradient(
                Q + self.rho * self.I,
                b=b + self.rho * (z - u),
                x=x,
                maxiter=self.maxiter_cg,
                disable_progressbar=disable_progressbar,
            )
            z = self.regularizer(x + u)
            u = u + x - z

        return x

## Linear operators

Complex tensors are handled as flattened real vectors (`view_as_real`), so the
operators act on real vectors of twice the length.

`ChannelOperator` maps a single image to multi-coil images (and back, via the
coil-combining adjoint).

In [ ]:
class ChannelOperator(LinearOperator):
    """Coil sensitivity operator: image to multi-coil images.

    Parameters
    ----------
    coil_sensitivities : torch.Tensor
        Complex coil sensitivities of shape ``(n_channels, *data_shape)``.
    data_shape : Tuple[int, ...]
        Spatial shape of the image.
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        n_channels = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (n_channels, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the operator: image to multi-coil images.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint: multi-coil images to coil-combined image.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened coil-combined image.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

`NonuniformFourierTransformOperator` maps multi-coil images to non-Cartesian
k-space samples along the given trajectory (and back, via the adjoint NUFFT).

In [ ]:
class NonuniformFourierTransformOperator(LinearOperator):
    """Non-Cartesian Fourier operator: multi-coil images to k-space samples.

    Parameters
    ----------
    k : torch.Tensor
        k-space trajectory of shape ``(n_dims, n_samples)``.
    input_shape : Tuple[int, ...]
        Shape ``(n_channels, n_x, n_y)`` of the multi-coil images.
    device : Optional[torch.device], optional
        Device on which the NUFFT objects are placed.
    """

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        _, n_samples = k.shape
        n_channels, n_x, n_y = input_shape

        self.n_modes = (n_x, n_y)
        self.k = k
        self.forward_shape = (n_channels, n_x, n_y, 1)
        self.adjoint_shape = (n_channels, n_samples)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the forward NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened k-space samples.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened k-space samples.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

## Encoding operator

Read the ADC k-space trajectory from the sequence file, normalise it to
$[-0.5, 0.5]$, and build the encoding operator $A = F C$ with $F$ the NUFFT and
$C$ the coil sensitivities.

In [ ]:
seq = pp.Sequence()
seq.read("Radial 2D GRE.seq")
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

F = NonuniformFourierTransformOperator(k_traj_adc, (n_channels, n_x, n_x))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_x))
A = F @ C

## CG reconstruction

Solve the normal equations $Q x = b$ with $Q = A^H A$ and $b = A^H d$ using 30 CG
iterations, starting from zero.

In [ ]:
k_data = signal.clone()
k_data = k_data.reshape((-1, n_channels)).permute((1, 0))
k_data = k_data.contiguous()

Q = A.H @ A
b = A.H @ k_data.view(torch.float32).ravel()
x_init = torch.zeros(n_x * n_x, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, b, x_init, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_x))

In [ ]:
plt.figure()
plt.imshow(torch.abs(x_recon_image).cpu().numpy().T, cmap="gray")
plt.show()

## Self-supervised training

Each step randomly splits the k-space samples into two halves: the model
reconstructs from the source half, and the loss (relative L1) compares the
predicted k-space on the target half with the measured data.

In [ ]:
shape = (n_x, n_x, 1)

torch.manual_seed(0)
model = DeepADMM(
    shape,
    rho_init=0.1,
    maxiter_admm=10,
    maxiter_cg=5,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

n_samples = k_traj_adc.shape[1]
num_splits = 100

model.train()
losses = []

progress_bar = tqdm(range(num_splits), desc="Self-supervised splits")

for split in progress_bar:
    np.random.seed(split)

    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    index_src, index_tar = np.split(indices, 2)

    k_traj_src = k_traj_adc[:, index_src]
    k_data_src = k_data[:, index_src].view(torch.float32).ravel()

    k_traj_tar = k_traj_adc[:, index_tar]
    k_data_tar = k_data[:, index_tar].view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_traj_src, (n_channels, n_x, n_x))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (n_channels, n_x, n_x))

    Q_src = C.H @ F_src.H @ F_src @ C
    b_src = C.H @ F_src.H @ k_data_src

    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)

    k_data_pred = F_tar @ C @ x_out

    target_norm = torch.linalg.norm(k_data_tar, ord=1)
    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / target_norm

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

## Trained reconstruction

Reconstruct from the full k-space data with the trained model.

In [ ]:
model.eval()
with torch.no_grad():
    x_deep_admm_trained = model(Q, b, b)

x_deep_admm_trained_image = x_deep_admm_trained.view(torch.complex64).reshape(
    (n_x, n_x)
)

In [ ]:
plt.imshow(torch.abs(x_deep_admm_trained_image).cpu().numpy().T, cmap="gray")
plt.title("DeepADMM (learned, trained)")
plt.show()

## Training loss

In [ ]:
plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("Split")
plt.ylabel("Self-supervised loss")
plt.title("DeepADMM training loss over splits")
plt.show()